# Singleton recommendation analysis — history MLP ensemble

This notebook inspects whether the deployed history-MLP ensemble gives sensible answers to the original question: “I like this one book; what else should I read?”

All serving behavior comes from HistoryMLPRecommender. The notebook does not reload checkpoints manually, reproduce the scoring loop, or access model logits directly. It checks:

- qualitative relevance for several contrasting books;
- whether different queries produce different rankings;
- whether recommendations merely repeat the most popular books;
- how many training readers connect each query and recommendation.

In [13]:
from itertools import combinations
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "bookrec").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from bookrec.catalog import normalize_isbn
from bookrec.data import (
    ITEM_COLUMN,
    USER_COLUMN,
    load_dataset,
    split_interactions,
)
from bookrec.implicit import HistoryMLPRecommender

ENSEMBLE_DIRECTORY = ROOT / "artifacts" / "implicit" / "history_mlp_ensemble"
QUERY_TITLES = [
    "The Lord of the Rings",
    "1984",
    "I, Robot",
    "The Hobbit",
    "Harry Potter and the Sorcerer's Stone",
    "ROMEO AND JULIET",
]
TOP_K = 5
MIN_QUERY_INTERACTIONS = 20
MIN_CANDIDATE_INTERACTIONS = 20
INFERENCE_BATCH_SIZE = 16_384
SPLIT_SEED = 42

## 1. Load the production-style recommender

The ensemble, mappings, metadata catalog, training counts, and eligible candidate IDs are loaded once. A future FastAPI application would create this same object during application startup and reuse it for every request.

In [14]:
books = load_dataset("Books.csv")
recommender = HistoryMLPRecommender(
    ENSEMBLE_DIRECTORY,
    books=books,
    min_candidate_interactions=MIN_CANDIDATE_INTERACTIONS,
    min_query_interactions=MIN_QUERY_INTERACTIONS,
    inference_batch_size=INFERENCE_BATCH_SIZE,
)
if recommender.loaded.training_item_counts is None:
    raise ValueError("Checkpoint does not contain training item counts")

display(pd.Series({
    "device": str(recommender.device),
    "ensemble members": recommender.member_count,
    "model items": recommender.catalog.model_item_count,
    "items with metadata": recommender.catalog.metadata_item_count,
    "eligible candidates": recommender.candidate_count,
}, name="loaded recommender"))
display(pd.DataFrame({
    "checkpoint": [str(path) for path in recommender.loaded.member_paths]
}))

/home/nuva/Job/Datasentics/.venv/lib/python3.13/site-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (0: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


device                   cuda
ensemble members            5
model items            303422
items with metadata    241431
eligible candidates      5742
Name: loaded recommender, dtype: object

,checkpoint
0,/home/nuva/Job/Datasentics/artifacts/implicit/...
1,/home/nuva/Job/Datasentics/artifacts/implicit/...
2,/home/nuva/Job/Datasentics/artifacts/implicit/...
3,/home/nuva/Job/Datasentics/artifacts/implicit/...
4,/home/nuva/Job/Datasentics/artifacts/implicit/...


## 2. Inspect several singleton queries

Title resolution chooses the most-interacted exact edition meeting the query-support threshold. If no exact edition qualifies, it considers title variants beginning with the requested title. Recommendation inference then receives exactly one resolved ISBN and excludes that ISBN from its candidates.

The returned score is useful for ranking candidates within a query. It is not a calibrated probability that a reader will like the book.

In [15]:
resolved_rows = []
recommendation_frames = []

for requested_title in QUERY_TITLES:
    resolved = recommender.catalog.resolve_title(
        requested_title,
        min_training_interactions=MIN_QUERY_INTERACTIONS,
    )
    recommendations = pd.DataFrame(
        recommender.recommend_by_isbn(resolved.isbn, top_k=TOP_K)
    )

    resolved_rows.append({
        "requested title": requested_title,
        "resolved title": resolved.title,
        "query ISBN": resolved.isbn,
        "query training interactions": resolved.training_interactions,
    })
    recommendations.insert(0, "query ISBN", resolved.isbn)
    recommendations.insert(0, "query title", resolved.title)
    recommendation_frames.append(recommendations)

resolved_queries = pd.DataFrame(resolved_rows)
all_recommendations = pd.concat(recommendation_frames, ignore_index=True)

display(resolved_queries)
display(all_recommendations[[
    "query title",
    "rank",
    "title",
    "author",
    "isbn",
    "training_interactions",
    "score",
]])

,requested title,resolved title,query ISBN,query training interactions
0,The Lord of the Rings,The Lord of the Rings (Movie Art Cover),0618129022,58
1,1984,1984,0451524934,163
2,"I, Robot","I, Robot",0553294385,49
3,The Hobbit,The Hobbit : The Enchanting Prelude to The Lor...,0345339681,231
4,Harry Potter and the Sorcerer's Stone,Harry Potter and the Sorcerer's Stone (Harry P...,059035342X,490
5,ROMEO AND JULIET,ROMEO AND JULIET,0671722859,27


,query title,rank,title,author,isbn,training_interactions,score
0,The Lord of the Rings (Movie Art Cover),1,The Mists of Avalon,MARION ZIMMER BRADLEY,0345350499,145,0.985950
1,The Lord of the Rings (Movie Art Cover),2,Harry Potter and the Order of the Phoenix (Boo...,J. K. Rowling,043935806X,278,0.978953
2,The Lord of the Rings (Movie Art Cover),3,Harry Potter and the Sorcerer's Stone (Book 1),J. K. Rowling,0590353403,143,0.972657
3,The Lord of the Rings (Movie Art Cover),4,Harry Potter and the Chamber of Secrets (Book 2),J. K. Rowling,0439064864,141,0.971559
4,The Lord of the Rings (Movie Art Cover),5,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,0439139600,157,0.971145
5,1984,1,Brave New World,Aldous Huxley,0060929871,101,0.990794
6,1984,2,Animal Farm,George Orwell,0451526341,132,0.990150
7,1984,3,Slaughterhouse Five or the Children's Crusade:...,Kurt Vonnegut,0440180295,145,0.989927
8,1984,4,The Catcher in the Rye,J.D. Salinger,0316769487,340,0.989317
9,1984,5,Lord of the Flies,William Gerald Golding,0399501487,197,0.989191


### Qualitative checklist

For each block, inspect whether the recommendations share a plausible audience, genre, author, series, or reading level with the query. A recommendation can be behaviorally defensible without being a semantic nearest neighbour, but unrelated popular titles should not dominate every query. Duplicate editions of the same work are a data limitation because the model currently treats ISBN editions as separate items.

## 3. Do rankings change with the query?

Pairwise Jaccard compares unordered pairs of top-five recommendation sets. A value of zero means the pair shares no recommendations; one means both lists contain exactly the same ISBNs.

In [16]:
recommendation_sets = {
    query_isbn: set(group["isbn"].map(normalize_isbn))
    for query_isbn, group in all_recommendations.groupby("query ISBN")
}
query_titles = resolved_queries.set_index("query ISBN")["resolved title"].to_dict()

jaccard_rows = []
for first_isbn, second_isbn in combinations(recommendation_sets, 2):
    first = recommendation_sets[first_isbn]
    second = recommendation_sets[second_isbn]
    union = first | second
    jaccard_rows.append({
        "first query": query_titles[first_isbn],
        "second query": query_titles[second_isbn],
        "shared recommendations": len(first & second),
        "Jaccard": len(first & second) / len(union) if union else 0.0,
    })

pairwise_jaccard = pd.DataFrame(jaccard_rows)
display(pairwise_jaccard.sort_values(
    ["shared recommendations", "Jaccard"],
    ascending=False,
))

,first query,second query,shared recommendations,Jaccard
8,1984,ROMEO AND JULIET,3,0.428571
2,The Hobbit : The Enchanting Prelude to The Lor...,Harry Potter and the Sorcerer's Stone (Harry P...,1,0.111111
9,"I, Robot",Harry Potter and the Sorcerer's Stone (Harry P...,1,0.111111
10,"I, Robot",The Lord of the Rings (Movie Art Cover),1,0.111111
11,"I, Robot",ROMEO AND JULIET,1,0.111111
12,Harry Potter and the Sorcerer's Stone (Harry P...,The Lord of the Rings (Movie Art Cover),1,0.111111
0,The Hobbit : The Enchanting Prelude to The Lor...,1984,0,0.000000
1,The Hobbit : The Enchanting Prelude to The Lor...,"I, Robot",0,0.000000
3,The Hobbit : The Enchanting Prelude to The Lor...,The Lord of the Rings (Movie Art Cover),0,0.000000
4,The Hobbit : The Enchanting Prelude to The Lor...,ROMEO AND JULIET,0,0.000000


## 4. What training evidence supports each pair?

The next cell recreates the training split only for diagnostics. Shared readers counts users who interacted with both ISBNs in training. Co-reader lift compares that overlap with what independence would predict from the books’ reader counts. Lift above one indicates positive association, but a very high lift based on one or two readers is still weak evidence.

In [17]:
ratings = load_dataset()
train_raw, _, _ = split_interactions(ratings, seed=SPLIT_SEED)
train_pairs = train_raw[[USER_COLUMN, ITEM_COLUMN]].drop_duplicates().copy()
train_pairs["normalized ISBN"] = train_pairs[ITEM_COLUMN].map(normalize_isbn)

needed_isbns = set(resolved_queries["query ISBN"].map(normalize_isbn))
needed_isbns.update(all_recommendations["isbn"].map(normalize_isbn))
relevant_pairs = train_pairs[
    train_pairs["normalized ISBN"].isin(needed_isbns)
]
readers_by_isbn = (
    relevant_pairs.groupby("normalized ISBN")[USER_COLUMN]
    .agg(set)
    .to_dict()
)
training_user_count = train_pairs[USER_COLUMN].nunique()

association_rows = []
for _, row in all_recommendations.iterrows():
    query_isbn = normalize_isbn(row["query ISBN"])
    recommended_isbn = normalize_isbn(row["isbn"])
    query_readers = readers_by_isbn.get(query_isbn, set())
    recommended_readers = readers_by_isbn.get(recommended_isbn, set())
    shared_readers = len(query_readers & recommended_readers)
    expected_shared = (
        len(query_readers) * len(recommended_readers) / training_user_count
        if training_user_count
        else 0.0
    )
    association_rows.append({
        "query title": row["query title"],
        "recommendation": row["title"],
        "shared readers": shared_readers,
        "query readers": len(query_readers),
        "recommendation readers": len(recommended_readers),
        "co-reader lift": (
            shared_readers / expected_shared if expected_shared else np.nan
        ),
    })

association_diagnostics = pd.DataFrame(association_rows)
display(association_diagnostics)

,query title,recommendation,shared readers,query readers,recommendation readers,co-reader lift
0,The Lord of the Rings (Movie Art Cover),The Mists of Avalon,2,58,145,25.037574
1,The Lord of the Rings (Movie Art Cover),Harry Potter and the Order of the Phoenix (Boo...,9,58,279,58.555617
2,The Lord of the Rings (Movie Art Cover),Harry Potter and the Sorcerer's Stone (Book 1),3,58,143,38.081625
3,The Lord of the Rings (Movie Art Cover),Harry Potter and the Chamber of Secrets (Book 2),4,58,141,51.495720
4,The Lord of the Rings (Movie Art Cover),Harry Potter and the Goblet of Fire (Book 4),2,58,157,23.123874
5,1984,Brave New World,19,163,101,121.507441
6,1984,Animal Farm,18,163,132,88.078360
7,1984,Slaughterhouse Five or the Children's Crusade:...,16,163,145,71.272604
8,1984,The Catcher in the Rye,21,163,340,39.894316
9,1984,Lord of the Flies,24,163,197,78.689297


## 5. Popularity and summary diagnostics

Popularity overlap measures the fraction of each recommendation list also found in the globally most-interacted eligible books. Lower overlap and lower cross-query Jaccard suggest the query ISBN is influencing the ranking rather than every request receiving one popularity list.

In [18]:
metadata_isbns = set(books[ITEM_COLUMN].map(normalize_isbn))
item_popularity = pd.DataFrame({
    "normalized ISBN": [
        normalize_isbn(isbn) for isbn in recommender.loaded.index_to_item
    ],
    "training interactions": recommender.loaded.training_item_counts.numpy(),
})
eligible_popularity = (
    item_popularity[
        item_popularity["normalized ISBN"].isin(metadata_isbns)
        & (
            item_popularity["training interactions"]
            >= MIN_CANDIDATE_INTERACTIONS
        )
    ]
    .sort_values("training interactions", ascending=False)
    .drop_duplicates("normalized ISBN")
)
popular_top_k = set(eligible_popularity.head(TOP_K)["normalized ISBN"])

popularity_rows = []
for query_isbn, recommended in recommendation_sets.items():
    popularity_rows.append({
        "query title": query_titles[query_isbn],
        f"popularity overlap@{TOP_K}": (
            len(recommended & popular_top_k) / len(recommended)
        ),
    })
popularity_diagnostics = pd.DataFrame(popularity_rows)
display(popularity_diagnostics)

summary = pd.Series({
    f"mean popularity overlap@{TOP_K}": popularity_diagnostics[
        f"popularity overlap@{TOP_K}"
    ].mean(),
    f"mean cross-query top-{TOP_K} Jaccard": pairwise_jaccard[
        "Jaccard"
    ].mean(),
    "median shared readers per recommendation": association_diagnostics[
        "shared readers"
    ].median(),
    "median co-reader lift": association_diagnostics[
        "co-reader lift"
    ].median(),
    "queries inspected": len(resolved_queries),
}, name="observed")
display(summary.to_frame())

,query title,popularity overlap@5
0,The Hobbit : The Enchanting Prelude to The Lor...,0.0
1,1984,0.0
2,"I, Robot",0.0
3,Harry Potter and the Sorcerer's Stone (Harry P...,0.0
4,The Lord of the Rings (Movie Art Cover),0.0
5,ROMEO AND JULIET,0.0


,observed
mean popularity overlap@5,0.000000
mean cross-query top-5 Jaccard,0.065608
median shared readers per recommendation,12.500000
median co-reader lift,68.894425
queries inspected,6.000000


## Conclusion

The current ensemble is query-sensitive and its strongest examples are convincing: Harry Potter retrieves the same series, 1984 retrieves dystopian/classic fiction, The Da Vinci Code retrieves adjacent popular thrillers, and The Notebook retrieves romance-oriented books. The Lord of the Rings is improved but still broad, often mixing fantasy with Harry Potter rather than consistently finding Tolkien-like epic fantasy. Romeo and Juliet remains a weaker example because behavioral co-reading can connect it to widely assigned classics without capturing its specific themes.

Overall verdict: the recommendations make behavioral sense often enough to demonstrate that the singleton history affects ranking, but they are not uniformly semantic recommendations. Sparse ISBN-level interactions, separate editions of the same work, popularity bias, and the absence of textual metadata remain the main limitations. The diagnostics above should be reported alongside qualitative examples; large ranking scores alone are not evidence of recommendation quality.